# 最终定稿：固定权重融合

先学会第 09 课的完整单模型，再学习如何利用互补误差。本课只保留最后真正使用的 5 个成员，不做调参和候选搜索。

## 1. 数据与成员配置

In [ ]:

from pathlib import Path
import sys
import pickle
import numpy as np
from sklearn.model_selection import KFold
HERE = Path.cwd()
if not (HERE/'members.json').exists():
    raise RuntimeError('请从第 10 课文件夹打开 Notebook。')
sys.path.insert(0, str(HERE.parent/'_shared'))
from course_utils import DATA_DIR, load_data, rmsle, save_outputs
from model import MEMBERS, make_features, fit_model, predict_log, best_rounds

data_dir=HERE.parent/'data'
output_dir=HERE/'outputs'


In [ ]:
import pandas as pd
display(pd.DataFrame(MEMBERS)[["key","family","weight","mode","seed"]])

In [ ]:

train, test = load_data(data_dir)
X = train.drop(columns=['id','result'])
T = test.drop(columns=['id'])
target = np.log1p(train.result.to_numpy())
folds = list(KFold(5, shuffle=True, random_state=42).split(X))


## 2. 一个成员如何训练
每个成员都是之前讲过的单算法流程：五折 OOF 选轮数，再训练一个全量模型。特征映射仍只在训练折拟合。

In [ ]:

def train_member(member, X, target, T, folds, output_dir):
    directory = Path(output_dir)/'checkpoints'/member['key']
    directory.mkdir(parents=True, exist_ok=True)
    oof_log = np.zeros(len(X))
    bag_log = np.zeros(len(T))
    rounds = []
    for fold, (train_index, valid_index) in enumerate(folds):
        features = make_features(member)
        A = features.fit_transform(X.iloc[train_index], target[train_index])
        B = features.transform(X.iloc[valid_index])
        estimator = fit_model(member, A, target[train_index], B, target[valid_index])
        oof_log[valid_index] = predict_log(estimator, B, member['family'])
        bag_log += predict_log(estimator, features.transform(T), member['family'])/len(folds)
        rounds.append(best_rounds(estimator, member['family']))
        for name, value in [('builder', features), ('model', estimator)]:
            with (directory/f'fold{fold}_{name}.pkl').open('wb') as file:
                pickle.dump(value, file)
        print(member['key'], 'fold', fold+1, 'rounds', rounds[-1])
    # 轮数来自验证；全量模型使用全部训练数据拟合。
    features = make_features(member)
    A = features.fit_transform(X, target)
    count = int(np.median(rounds))
    estimator = fit_model(member, A, target, rounds=count)
    full_log = predict_log(estimator, features.transform(T), member['family'])
    for name, value in [('builder', features), ('model', estimator)]:
        with (directory/f'full_{name}.pkl').open('wb') as file:
            pickle.dump(value, file)
    return oof_log, 0.5*bag_log+0.5*full_log, count


## 3. 冻结权重融合
先在每个成员内部做 50% 折模型平均 + 50% 全量模型，再按 members.json 权重融合。全部在 log1p 空间进行。

In [ ]:

oof_log = np.zeros(len(train))
test_log = np.zeros(len(test))
training_rounds = {}
for member in MEMBERS:
    member_oof, member_test, count = train_member(member, X, target, T, folds, output_dir)
    # 权重是冻结的历史开发结果，本课不再读取榜单调权重。
    oof_log += member['weight']*member_oof
    test_log += member['weight']*member_test
    training_rounds[member['key']] = count


## 4. 评分并保存
权重历史上利用过这些 OOF，不能把本课 OOF 作为完全独立测试。

In [ ]:

oof = np.clip(np.expm1(oof_log),0,1)
predictions = np.clip(np.expm1(test_log),0,1)
score = rmsle(train.result, oof)
metrics = {'oof_rmsle':score, 'rounds':training_rounds,
           'weights':'members.json 中的冻结权重',
           'note':'这些权重历史上使用过相同验证划分，OOF 不是独立盲测。'}
save_outputs(output_dir, train, test, oof, predictions, metrics)
print('本次 OOF RMSLE:', score)


## 5. 查看已经完成的线上记录
下面是历史成绩，不是本次重训的线上成绩。

In [ ]:
import json
json.loads((HERE/'reference/scores.json').read_text())

## 课堂讨论
1. 为什么预测高度相关，误差仍可能互补？
2. 为什么 OOF 选出的权重不等于最优 Private 权重？
3. 30 个模型只比 CatBoost 单模型好 0.00017，什么场景下值得采用？
4. 为什么我们不选择同样 Private、模型更多的剪枝融合？